# Шаг 2. YOLO смотрит в веб-камеру в реальном времени

В первом шаге модель разбирала одну фотографию. Видео — это просто **много фотографий (кадров) подряд**.
Если успевать обрабатывать кадры достаточно быстро, получится распознавание «вживую».

Что мы сделаем:
1. Загрузим ту же модель, что и в шаге 1.
2. Проверим, что камера работает: снимем один кадр и посмотрим на него прямо в ноутбуке.
3. Запустим цикл: взять кадр → распознать → нарисовать рамки → показать под ячейкой → повторить.
4. Будем измерять скорость (FPS) и выводить список найденных объектов на русском.

> **Это версия для Google Colab.** Кадры снимает браузер и передаёт на сервер Colab, поэтому видео идёт
> медленнее и рывками: это нормально. Когда браузер спросит про камеру, нажмите «Разрешить».
> Для скорости включите GPU: «Среда выполнения → Сменить среду выполнения». На своём компьютере откройте `02_webcam.ipynb`.

In [ ]:
# Настройка для Google Colab. На своём компьютере эта ячейка ничего не делает.
# В Colab нужна папка machine_vision на Google Диске (в ней session01–session05 и dataset).
import os
import sys

try:
    from google.colab import drive
except ImportError:
    drive = None

if drive:
    drive.mount("/content/drive")
    COLAB_ROOT = "/content/drive/MyDrive/machine_vision"
    os.chdir(f"{COLAB_ROOT}/session01")  # здесь лежит common.py и сохраняются результаты
    sys.path.insert(0, os.getcwd())
    import subprocess

    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics"], check=True)

## 1. Импорт библиотек

- `cv2` (библиотека **OpenCV**) — работа с картинками и рисование на них.
- `time` — замер времени, чтобы посчитать скорость.
- `Counter` — удобный «счётчик»: посчитает, сколько людей, сколько чашек и т.д.
- `YOLO` и `ru` — как в шаге 1.
- `colab_camera` — камера через браузер: `frames` выдаёт кадры, `show` показывает их, `snapshot` снимает один кадр.

In [ ]:
import time
from collections import Counter

import cv2
from ultralytics import YOLO

from colab_camera import frames, show, snapshot
from labels_ru import ru

## 2. Модель

In [ ]:
MODEL = "yolov8n.pt"

model = YOLO(MODEL)

## 3. Проверяем камеру: один кадр

Прежде чем запускать цикл, убедимся, что камера вообще работает.

`snapshot()` включает камеру, снимает один кадр `frame` и сразу выключает её.

Кадр распознаём так же, как фото в шаге 1, и показываем прямо в ноутбуке.

In [ ]:
from PIL import Image

frame = snapshot()

result = model(frame, conf=0.40, verbose=False)[0]
Image.fromarray(result.plot()[..., ::-1])  # BGR -> RGB, как в шаге 1

## 4. Распознавание почти в реальном времени

Теперь главное — цикл, который крутится, пока мы не нажмём кнопку «Стоп». На каждом проходе:

1. **Берём кадр** с камеры: цикл `for frame, pressed in frames(...)` выдаёт кадр и название нажатой кнопки.
2. **Распознаём** его моделью. `verbose=False` отключает подробный вывод, иначе ноутбук завалит текстом.
3. **Рисуем рамки** — `result.plot()`.
4. **Считаем FPS** (frames per second, кадров в секунду): засекаем, сколько секунд прошло с прошлого кадра,
   и делим 1 на это время. Например, 0.25 с на кадр → 4 FPS. Пишем FPS и текущий порог прямо на картинке (`cv2.putText`).
   В Colab это время включает путь кадра через интернет, поэтому FPS ниже, чем на своём компьютере.
5. **Пишем, что в кадре**, по-русски, например `человек: 2, чашка: 1`. Чтобы не печатать одно и то же
   на каждом кадре, выводим строку только когда **состав меняется**.
6. **Показываем кадр** под ячейкой (`show`).
7. **Проверяем кнопки** под видео:

| Кнопка | Действие |
|---------|----------|
| `+` / `-` | поднять / опустить порог уверенности на 0.05 |
| `Сохранить` | сохранить кадр в файл `shot_01.jpg`, `shot_02.jpg`… рядом с ноутбуком (пригодится для таблицы ошибок) |
| `Стоп` | выход |

Камеру выключает сама функция `frames`, когда цикл заканчивается: и по кнопке «Стоп», и после ошибки.

In [ ]:
def run_webcam(conf=0.40):
    shot = 0
    last_report = ""
    prev = time.time()

    # 1. Берём кадр и нажатую кнопку
    for frame, pressed in frames(buttons=["-", "+", "Сохранить"]):
        # 2–3. Распознаём и рисуем рамки с подписями
        result = model(frame, conf=conf, verbose=False)[0]
        picture = result.plot()

        # 4. Скорость: сколько кадров в секунду успевает пройти весь путь
        now = time.time()
        fps = 1 / (now - prev)
        prev = now
        cv2.putText(picture, f"FPS {fps:4.1f}   conf {conf:.2f}", (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)

        # 5. Что в кадре — по-русски, только когда состав меняется
        counts = Counter(result.names[int(c)] for c in result.boxes.cls)
        report = ", ".join(f"{ru(n)}: {k}" for n, k in counts.most_common())
        if report != last_report:
            print(report or "никого")
            last_report = report

        # 6. Показываем кадр
        show(picture)

        # 7. Кнопки
        if pressed == "+":
            conf = min(conf + 0.05, 0.95)
        if pressed == "-":
            conf = max(conf - 0.05, 0.05)
        if pressed == "Сохранить":
            shot += 1
            cv2.imwrite(f"shot_{shot:02d}.jpg", picture)
            print(f"Сохранено: shot_{shot:02d}.jpg")

Запускаем! Под ячейкой появится видео с рамками и кнопки. Чтобы выйти, нажмите «Стоп».

In [ ]:
run_webcam(conf=0.40)

## Попробуйте сами

1. Посмотрите на FPS. Сколько кадров в секунду получается в Colab? Замените модель на `yolov8s.pt` — как изменится скорость?
2. Опускайте порог кнопкой `-`, пока не появятся ошибки. При каком пороге модель начинает «видеть» то, чего нет?
3. Покажите камере предметы под разными углами, частично закройте их рукой. Когда модель перестаёт их узнавать?
   Сохраняйте такие кадры кнопкой «Сохранить» — это материал для таблицы ошибок.
4. Измените функцию так, чтобы она печатала сообщение, когда в кадре появляется **телефон** (`"cell phone"`).